# Learn Milvus: embeddings, collections, and retrieval

Run the cells from top to bottom using **Run → Run All Cells** in JupyterLab.

This demo uses **Milvus Lite**, a local database file, and a real **Sentence Transformers** embedding model. No cloud account or API key is needed. The first run downloads the model; later runs reuse the local cache.

You will create a collection, embed support articles, store them, search by meaning, filter results, and retrieve exact records. The final section explains server-only aliases and Attu.

**Flow:** text → embedding model → vectors + text + metadata in Milvus. Question → same model → nearest matches.

### Before you start

A notebook is a page containing notes and small blocks of Python called **cells**. Run them in order because later cells use things created earlier. Use **Shift + Enter** to run one cell.

We are building a tiny support-article search. You ask a question, and it finds useful articles even when the words are different. Milvus Lite runs on your laptop. No company database is used.

The next cell imports two tools: `SentenceTransformer` turns text into numbers; `MilvusClient` lets Python talk to Milvus. `DB_PATH` is where your database is saved. The printed path tells you which database this run uses.


In [1]:
from pathlib import Path
import os

# Find the project folder from the original notebook or the verified copy.
current_folder = Path.cwd().resolve()
PROJECT_ROOT = next(
    folder for folder in (current_folder, *current_folder.parents)
    if (folder / "pyproject.toml").is_file()
)
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(PROJECT_ROOT / ".cache" / "huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer

DB_PATH = Path(os.environ.get("MILVUS_DEMO_DB", str(DATA_DIR / "milvus_learning.db"))).expanduser().resolve()
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
COLLECTION = "support_articles_v1"
print("Database:", DB_PATH)


/Users/prafullsaxena/Desktop/Learning/Machine Learning/milvus/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Database: /Users/prafullsaxena/Desktop/Learning/Machine Learning/milvus/data/milvus_verification.db


## 1. Generate real embeddings

The model converts each text into a list of numbers. This model produces **384 numbers per text**. Use the same model for stored documents and search questions. We use CPU for a portable demo.

### What is an embedding?

An **embedding** is a list of numbers that represents a piece of text. A **vector** is the mathematical name for that list. A model has already learned patterns from text, so sentences with similar meanings often get similar vectors. A single number does not have a simple label such as “password” or “invoice”.

For example, “I forgot my password” and “Help me reset my password” should have similar vectors. “Where is my package?” should be less similar. This is useful because matching words exactly would miss many related questions.

### Which model are we using?

We use **`sentence-transformers/all-MiniLM-L6-v2`**, a pretrained text embedding model. **Sentence Transformers** is the Python library that loads and runs it. We are using an existing model, not training a new one. It runs locally on the CPU and does not need an API key.

### What does the code do?

1. `SentenceTransformer(...)` loads the model (and downloads it on the first run).
2. `articles` contains six short example articles. Each has an ID, text, and category.
3. `model.encode(...)` sends those sentences through the model and returns their embeddings. **This line creates the embeddings; Milvus does not create them in this demo.**
4. `normalize_embeddings=True` scales each vector to length 1. Its direction stays the same, which works well for comparing cosine similarity.
5. The shape should be **`(6, 384)`**: six articles, each represented by 384 numbers.

Each article is already short, so we use the whole sentence. For a long document, you would first split it into smaller pieces called **chunks**, then embed each chunk.


In [2]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME, device="cpu")
dimension = model.get_sentence_embedding_dimension()
print("Embedding dimensions:", dimension)

articles = [
    {"id": 1, "text": "Reset your password using the forgot password link on the login page.", "category": "account"},
    {"id": 2, "text": "Unlock your account by contacting the support team after too many failed login attempts.", "category": "account"},
    {"id": 3, "text": "Enable two-factor authentication to protect your account.", "category": "account"},
    {"id": 4, "text": "Change your billing address from the payments settings page.", "category": "billing"},
    {"id": 5, "text": "Download your monthly invoice from the billing dashboard.", "category": "billing"},
    {"id": 6, "text": "Track your package using the shipment tracking number in your order email.", "category": "shipping"},
]
vectors = model.encode([a["text"] for a in articles], normalize_embeddings=True)
print("Shape (articles, dimensions):", vectors.shape)
print("First 8 numbers of the first embedding:", vectors[0][:8])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7545.69it/s]

/var/folders/36/2jhn_y011tz6njws5pdnxys40000gn/T/ipykernel_51701/3504170191.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = model.get_sentence_embedding_dimension()


Embedding dimensions: 384


Shape (articles, dimensions): (6, 384)
First 8 numbers of the first embedding: [ 0.01580772 -0.06741775  0.00759386  0.01247069  0.03191539  0.06377517
 -0.03736277 -0.04508867]


## 2. Create a collection

A **collection** is similar to a table. An **entity** is one row. Our rows have `id`, `vector`, `text`, and `category`.

Quick setup creates the `id` and `vector` fields and permits additional metadata fields. `COSINE` measures similarity; higher scores mean closer matches. The vector dimension must match the model. Quick setup handles indexing and loading.

The guard below preserves an existing collection. If you change the embedding model, use a new collection name and regenerate document embeddings.

### Think of this as a table

| Field | What it stores | Why we keep it |
|---|---|---|
| `id` | A unique number for the article | Find or update that exact article |
| `vector` | The 384 numbers from the model | Compare meanings |
| `text` | The original sentence | Show a readable result |
| `category` | A label such as `billing` | Narrow down searches |

A **schema** describes the fields and their types. Extra information such as `category` is called **metadata**. An **index** helps the database look up vectors.

`has_collection()` asks whether our table already exists. We create it only when it is missing, so a second run keeps the saved data.

**Creating and loading are different.** Creating makes the collection. Loading gets it ready for searches. A saved collection can exist but be in a `released` state after restarting the local database. That is why we call `load_collection()` every time, outside the `if` block. Loading does not regenerate embeddings or insert articles.

`COSINE` compares vector directions; higher scores mean greater similarity. `Strong` asks searches to see completed writes.


In [3]:
client = MilvusClient(uri=str(DB_PATH))
if not client.has_collection(collection_name=COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        dimension=dimension,
        metric_type="COSINE",
        consistency_level="Strong",
    )

# Existing collections must also be loaded after reopening the database.
client.load_collection(collection_name=COLLECTION)
print("Load state:", client.get_load_state(collection_name=COLLECTION))
print("Collections:", client.list_collections())
client.describe_collection(collection_name=COLLECTION)


Load state: {'state': <LoadState: Loaded>}
Collections: ['support_articles_v1']


{'collection_name': 'support_articles_v1',
 'auto_id': False,
 'num_shards': 1,
 'description': '',
 'fields': [{'field_id': 0,
   'name': 'id',
   'description': '',
   'type': <DataType.INT64: 5>,
   'params': {},
   'is_primary': True},
  {'field_id': 0,
   'name': 'vector',
   'description': '',
   'type': <DataType.FLOAT_VECTOR: 101>,
   'params': {'dim': 384}}],
 'functions': [],
 'aliases': [],
 'collection_id': 0,
 'consistency_level': 0,
 'consistency_level_name': 'Strong',
 'properties': {},
 'num_partitions': 1,
 'enable_dynamic_field': True,
 'enable_namespace': False}

## 3. Store vectors alongside text and metadata

`insert()` adds records. Here we use **`upsert()`** so rerunning the notebook replaces records with the same IDs instead of adding them again. Data persists in `data/milvus_learning.db` after the notebook closes.

The model returned arrays of numbers. `.tolist()` turns each array into a normal Python list for the database. `rows` combines each vector with its article's ID, text, and category.

**Upsert means “update if the ID already exists; insert if it does not”.** Running this cell again updates the same six articles. It does not create six new IDs. The response tells you how many records were written.

We store the original text too because an embedding is not a readable copy of the sentence. Milvus returns the stored text when we ask for it.


In [4]:
rows = [dict(article, vector=vector.tolist()) for article, vector in zip(articles, vectors)]
write_result = client.upsert(collection_name=COLLECTION, data=rows)
print(write_result)


{'upsert_count': 6, 'ids': [1, 2, 3, 4, 5, 6]}


## 4. Search by meaning

Embed the question and request the top three matches. `data` is a list of query vectors, so the response contains one result list per query. `output_fields` controls which stored fields are returned.

Milvus names the score `distance`, even for COSINE similarity. It is **not a probability or confidence percentage**. Search returns the nearest records even when none is useful.

### Follow one question through the code

1. `question` contains what the user typed.
2. `model.encode(question, ...)` turns that question into 384 numbers using the **same model** as the articles.
3. `client.search(...)` asks Milvus which stored article vectors are closest.
4. `limit=3` means “return the best three matches”. This is also called **top-k**, where k is 3.
5. `output_fields` asks Milvus to return the original text and category with each match.
6. `results[0]` selects the answers for our first (and only) question.

You should see the password-reset article near the top. The printed score measures vector similarity; it does not mean the answer is correct with that percentage of certainty.

Milvus is returning stored articles here. It is not writing a new answer. In a RAG app, you could pass these articles to a separate language model to help it write an answer.


In [5]:
question = "I forgot my password. How can I sign in again?"
query_vector = model.encode(question, normalize_embeddings=True).tolist()
results = client.search(
    collection_name=COLLECTION,
    data=[query_vector],
    limit=3,
    output_fields=["text", "category"],
)

def show_hits(hits):
    for rank, hit in enumerate(hits, start=1):
        print(f"{rank}. ID={hit['id']} | cosine={hit['distance']:.4f} | {hit['entity']['category']}")
        print("  ", hit["entity"]["text"])

show_hits(results[0])


1. ID=1 | cosine=0.7273 | account
   Reset your password using the forgot password link on the login page.
2. ID=2 | cosine=0.5843 | account
   Unlock your account by contacting the support team after too many failed login attempts.
3. ID=3 | cosine=0.4734 | account
   Enable two-factor authentication to protect your account.


## 5. Combine metadata filtering with similarity search

This search considers only records whose category is `account`, then returns the closest matches. In real applications, filters can restrict results by document type, language, or tenant. Application permissions still need to be enforced correctly.

Think of a filter as a rule: **“Only consider account articles.”** The vector search then chooses the closest matches from those allowed articles.

`filter='category == "account"'` sets the rule, and `limit=2` asks for two results. Every printed result should have category `account`. The label is stored metadata; the embedding model did not invent it.


In [6]:
filtered_results = client.search(
    collection_name=COLLECTION,
    data=[query_vector],
    filter='category == "account"',
    limit=2,
    output_fields=["text", "category"],
)
show_hits(filtered_results[0])


1. ID=1 | cosine=0.7273 | account
   Reset your password using the forgot password link on the login page.
2. ID=2 | cosine=0.5843 | account
   Unlock your account by contacting the support team after too many failed login attempts.


## 6. Retrieve exact records without embeddings

- **`search()`**: nearest vectors, ranked by similarity.
- **`query()`**: records satisfying a metadata expression.
- **`get()`**: records with specific primary IDs.

Use `search()` for **“Which articles mean something like my question?”**

Use `query()` for **“Give me articles whose category is billing.”** This is an exact rule, so no embedding is needed. You should get IDs 4 and 5.

Use `get()` for **“Give me article number 1.”** You already know the record ID, so no similarity comparison is needed.


In [7]:
billing_articles = client.query(
    collection_name=COLLECTION,
    filter='category == "billing"',
    output_fields=["text", "category"],
)
print("Billing articles:")
for article in billing_articles:
    print(article)

print("\nRecord with ID 1:")
print(client.get(collection_name=COLLECTION, ids=[1], output_fields=["text", "category"]))


Billing articles:
{'id': 4, 'text': 'Change your billing address from the payments settings page.', 'category': 'billing'}
{'id': 5, 'text': 'Download your monthly invoice from the billing dashboard.', 'category': 'billing'}

Record with ID 1:
data: ["{'id': 1, 'text': 'Reset your password using the forgot password link on the login page.', 'category': 'account'}"], extra_info: {}


## 7. Check that the demo worked

These checks verify actual stored records, metadata filtering, and retrieval of a known article using its own embedding. The semantic results above are also useful to inspect manually.

An `assert` is a small check: Python stops with an error if the condition is false. These checks confirm that all six records are present, the billing filter finds the right IDs, and an article's own vector finds that article.

If everything works, the cell prints **“Passed: storage, metadata query, filtered search, and vector retrieval.”** This checks the demo's basic behaviour; it does not prove that every possible user question will get a useful answer.


In [8]:
stored = client.get(collection_name=COLLECTION, ids=[a["id"] for a in articles], output_fields=["text"])
assert len(stored) == len(articles), "Some demo records were not retrieved"
assert {row["id"] for row in billing_articles} == {4, 5}
assert len(filtered_results[0]) == 2
assert all(hit["entity"]["category"] == "account" for hit in filtered_results[0])
self_match = client.search(collection_name=COLLECTION, data=[vectors[0].tolist()], limit=1)
assert self_match[0][0]["id"] == 1, "The first article should match its own vector"
print("Passed: storage, metadata query, filtered search, and vector retrieval.")


Passed: storage, metadata query, filtered search, and vector retrieval.


## 8. Try your own question

Change the question below and rerun this cell. Try “Where can I find my invoice?” or “Where is my delivery?” Observe how the top result changes.

For the invoice question, expect article 5 near the top. Also try an unrelated question such as “How do I grow tomatoes?” Milvus will still return nearby articles from this tiny dataset. A real search app needs a way to recognise when its results are not useful.

If you already ran the final cleanup cell, rerun the connection/collection cell before this exercise.


In [9]:
my_question = "Where can I find my invoice?"
my_vector = model.encode(my_question, normalize_embeddings=True).tolist()
my_results = client.search(
    collection_name=COLLECTION,
    data=[my_vector],
    limit=3,
    output_fields=["text", "category"],
)
show_hits(my_results[0])


1. ID=5 | cosine=0.6432 | billing
   Download your monthly invoice from the billing dashboard.
2. ID=6 | cosine=0.2670 | shipping
   Track your package using the shipment tracking number in your order email.
3. ID=4 | cosine=0.2578 | billing
   Change your billing address from the payments settings page.


## 9. Aliases and Attu: use a Milvus server

**Milvus Lite does not support collection aliases.** The following is reference code, deliberately not an executable cell. Run it only against your own development server after creating the named collections.

```python
import os
server = MilvusClient(
    uri=os.environ["MILVUS_URI"],
    token=os.environ["MILVUS_TOKEN"],
)
server.create_alias(collection_name="support_articles_v1", alias="support_articles")
# Application searches can now use collection_name="support_articles".

# Only after v2 is created, populated, indexed, and loaded:
server.alter_alias(collection_name="support_articles_v2", alias="support_articles")
```

An alias is a pointer, not a copy: `support_articles → support_articles_v1`. You can later point it at v2 without changing the name your application searches. A model change also requires compatible query embeddings.

**Attu** is a separate graphical management application that connects to a Milvus server. Ask your team for its development Attu URL and Milvus version. In Attu, inspect the collection schema, records, indexes, aliases, and load status. This notebook uses an embedded local file, not an Attu server setup.

## 10. Close the connection

Run this after experimenting. If you want to search again afterward, rerun the connection cell first. Closing preserves your data. This cell also stops this database's embedded server to release its lock. If a run fails before cleanup, restart or shut down its kernel before opening the same database from another process.

### A simple way to remember aliases and the UI

An **alias** is a nickname for a collection. Your app asks for `support_articles`, and Milvus looks in whichever collection that nickname points to. Before moving the nickname to a new collection, fill that collection with data and make it ready to search.

**Attu** is a visual control panel: you click to inspect the Milvus server's collections and records. **Jupyter** is where we run Python and learn step by step. They serve different purposes.

The final Python cell closes this demo's connection and stops its embedded server. Stopping releases the database lock so another process can open the same path. It does not delete your articles.


In [10]:
# Closing the client alone does not stop the embedded server in this version.
client.close()
from milvus_lite.server_manager import server_manager_instance
server_manager_instance.release_server(str(DB_PATH))
print("Connection closed and database lock released. Data remains at:", DB_PATH)


Connection closed and database lock released. Data remains at: /Users/prafullsaxena/Desktop/Learning/Machine Learning/milvus/data/milvus_verification.db


## Where to go next

1. Replace the sample articles with document chunks, retaining a document ID and source.
2. Experiment with questions that have no good answer; inspect why top-k alone is insufficient.
3. Connect to a development Milvus server and explore the same collection in Attu.
4. Practise switching an alias between two prepared collections.

References: [Milvus quickstart](https://milvus.io/docs/quickstart.md), [Milvus Lite limitations](https://milvus.io/docs/milvus_lite.md), [aliases](https://milvus.io/docs/manage-aliases.md), [Attu](https://github.com/zilliztech/attu), [Sentence Transformers](https://www.sbert.net/docs/quickstart.html).
